In [1]:
%pip install gymnasium
%pip install gymnasium[classic-control]

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import tensorflow as tf
import gymnasium as gym
from tensorflow.keras import layers
from collections import deque

In [3]:
# hyperparameters
GAMMA = 0.99
BATCH_SIZE = 64
MEMORY_SIZE = 1000000
EXPLORE_MAX = 1.0
EXPLORE_MIN = 0.1
EXPLORE_DECAY = 0.995

In [4]:
class DQN():
    def __init__(self, observation_space, action_space):
        self.action_space = action_space
        self.memory = deque(maxlen=MEMORY_SIZE)
        self.explare_rate = EXPLORE_MAX
        self.model = tf.keras.Sequential([
            layers.Dense(24, input_shape=(observation_space, ), activation='relu'),
            layers.Dense(24, activation='relu'),
            layers.Dense(action_space, activation='linear')
        ])
        self.model.compile(optimizer='adam', loss='mse')
    
    def act(self, state):
        if np.random.rand() < self.explare_rate:
            return np.random.choice(self.action_space)
        q_values = self.model.predict(state, verbose=0)
        return np.argmax(q_values[0])
    
    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def replay(self):
        if len(self.memory) < BATCH_SIZE:
            return
        batch = np.random.choice(len(self.memory), BATCH_SIZE, replace=False)
        for i in batch:
            state, action, reward, next_state, done = self.memory[i]
            q_update = reward
            if not done:
                q_update = reward + GAMMA * np.amax(self.model.predict(next_state, verbose=0)[0])
            q_values = self.model.predict(state, verbose=0)
            q_values[0][action] = q_update
            self.model.fit(state, q_values, verbose=0)
        self.explare_rate *= EXPLORE_DECAY
        self.explare_rate = max(EXPLORE_MIN, self.explare_rate)

In [5]:
env = gym.make('CartPole-v1', render_mode='human')
OBSERVATION_SPACE = env.observation_space.shape[0]
ACTION_SPACE = env.action_space.n
agent = DQN(OBSERVATION_SPACE, ACTION_SPACE)

c:\Users\jorda\Code\deep-learning\.venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
for run in range(1, 100):
    state, _ = env.reset()
    state = np.array(state)
    state = np.reshape(state, [1, OBSERVATION_SPACE])
    
    step = 0
    while True:
        step += 1
        action = agent.act(state)
        next_state, reward, done, info, _ = env.step(action)
        reward = reward if not done else -reward
        next_state = np.reshape(next_state, [1, OBSERVATION_SPACE])
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        if done:
            print(f"Run: {run}, Explore Rate: {agent.explare_rate:.2f}, Score: {step}")
            break
        agent.replay()
env.close()

c:\Users\jorda\Code\deep-learning\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Run: 1, Explore Rate: 1.00, Score: 39
Run: 2, Explore Rate: 1.00, Score: 16


KeyboardInterrupt: 

: 